# Experiment 009: Llama 3.2-1B-Instruct — Full Fine-Tune on 100K Routed Data

**Model:** meta-llama/Llama-3.2-1B-Instruct (1.24B params, all trainable)

**Method:** Full fine-tune (no LoRA — proven insufficient for grammar learning)

**Data:** ~95K train / ~5K eval — balanced 30% ADD, 30% SUB, 15% BETWEEN, 25% NO_ROUTE

**Key improvements over Exp 008b (SmolLM2-360M, 2.1% E2E):**
1. **Larger model** — 3.4× more params (1.24B vs 362M), much more capacity for routing grammar
2. **Formulaic templates** — 28 new templates matching validation harness patterns
3. **Hard negatives** — 991 external (ComplexTempQA) + 12 synthetic temporal-sounding non-computable questions
4. **Edge cases** — 10% boundary-biased time sampling (midnight rollover, noon, minute carry)
5. **5K steps** — reduced from 10K to mitigate catastrophic forgetting

---

⚙️ **Runtime:** Runtime → Change runtime type → **L4 GPU** (24GB VRAM)

L4 is required for 1B full FT. VRAM usage: ~15GB with gradient checkpointing. T4 (16GB) is too tight. A100 is overkill for 1B.

## 1. Setup & Dependencies

In [ ]:
!pip install -q transformers datasets accelerate torch
!pip install -q tensorboard bitsandbytes

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("No GPU found! Change runtime to L4 GPU.")

## 2. Authenticate with Hugging Face

Llama 3.2-1B-Instruct is a **gated model**. You need:
1. A Hugging Face account
2. Accept the Llama 3.2 license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
3. A HF access token with `read` scope

In [ ]:
from huggingface_hub import login

# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Authenticated via Colab Secret")
except Exception:
    # Option 2: Paste token manually
    print("⚠️ Could not auto-login. Use login(token='hf_...') manually.")
    login()

## 3. Clone Repo & Generate Data

JSONL data files are gitignored (too large). We regenerate them on Colab
using the committed generator and hard-negative extractor.

Steps:
1. Clone repo + install Node.js deps (for `generator_route.js`)
2. Extract hard negatives from ComplexTempQA
3. Generate 100K training samples (30/30/15/25 distribution)
4. Format for Llama's chat template via `format_for_model.py`

In [ ]:
import os, subprocess

if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
    print("✅ Repo cloned")
else:
    !cd tagzeit-gemma-2b && git pull
    print("✅ Repo updated")

# Install Node deps for the generator
!cd tagzeit-gemma-2b && npm install --silent
print("✅ Node dependencies installed")

In [ ]:
# Step 1: Extract hard negatives from ComplexTempQA
!cd tagzeit-gemma-2b && python tools/extract_hard_negatives.py \
    --output data/hard_negatives/complex_tempqa_noroute.jsonl \
    --count 7000

In [ ]:
# Step 2: Generate 100K training samples
!cd tagzeit-gemma-2b && node experiments/temporal-tagzeit/src/generator_route.js \
    --count 100000 \
    --output data/train/train_routed_009.jsonl \
    --eval data/eval/eval_routed_009.jsonl \
    --hard-negatives data/hard_negatives/complex_tempqa_noroute.jsonl

In [ ]:
# Step 3: Format for Llama 3.2-1B chat template
RAW_TRAIN = 'tagzeit-gemma-2b/data/train/train_routed_009.jsonl'
RAW_EVAL  = 'tagzeit-gemma-2b/data/eval/eval_routed_009.jsonl'
FORMATTED_TRAIN = 'tagzeit-gemma-2b/data/train/train_routed_009_llama1b.jsonl'
FORMATTED_EVAL  = 'tagzeit-gemma-2b/data/eval/eval_routed_009_llama1b.jsonl'

!cd tagzeit-gemma-2b && python tools/format_for_model.py \
    --model_id meta-llama/Llama-3.2-1B-Instruct \
    --input {RAW_TRAIN} \
    --output {FORMATTED_TRAIN} \
    --eval_input {RAW_EVAL} \
    --eval_output {FORMATTED_EVAL}

# Verify
fmt_train = int(subprocess.check_output(['wc', '-l', FORMATTED_TRAIN]).split()[0])
fmt_eval  = int(subprocess.check_output(['wc', '-l', FORMATTED_EVAL]).split()[0])
print(f"\nFormatted train: {fmt_train} samples")
print(f"Formatted eval:  {fmt_eval} samples")

In [ ]:
# Spot-check: verify the chat template looks correct
import json
with open(FORMATTED_TRAIN) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        rec = json.loads(line)
        print(f"--- Sample {i+1} ---")
        print(rec['text'][:400])
        print()

In [ ]:
# Mount Google Drive — checkpoints survive Colab disconnects
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/tagzeit-009"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Google Drive mounted — checkpoints → {OUTPUT_DIR}")

## 4. Load Model & Tokenizer

**Key differences from Exp 008 (SmolLM2):**
- bf16 instead of float32 (Llama was trained in bf16, L4 has native bf16)
- `gradient_checkpointing=True` to fit 1.24B params in 24GB VRAM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token:  {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"\nGPU memory used (model only): {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Add Domain Tokens

Register the 104 routing tokens and initialise with geometric sinusoidal scheme.
Without this, the model can't produce `[ROUTE_*]` tokens.

In [ ]:
import math
import numpy as np

ROUTE_TOKENS = [
    "[ROUTE]", "[/ROUTE]", "[NO_ROUTE]",
    "[ROUTE_TIME_ADD]", "[ROUTE_TIME_SUB]", "[ROUTE_DURATION_BETWEEN]",
    "[ROUTE_CALENDAR_SHIFT]", "[ROUTE_TIMEZONE_CONVERT]",
    "[HEAD_TIME]", "[HEAD_DURATION]", "[HEAD_DATE]",
    "[REF_1]", "[REF_2]", "[REF_3]",
]
ROUTE_TOKENS += [f"[ARG_HOUR_{str(h).zfill(2)}]" for h in range(24)]
ROUTE_TOKENS += [f"[ARG_MIN_{str(m).zfill(2)}]" for m in range(60)]
ROUTE_TOKENS += ["[ARG_MON]", "[ARG_TUE]", "[ARG_WED]", "[ARG_THU]", "[ARG_FRI]", "[ARG_SAT]", "[ARG_SUN]"]

print(f"Domain tokens to add: {len(ROUTE_TOKENS)}")

num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added} tokens. New vocab size: {len(tokenizer)}")

embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        token_id = new_start + i
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02
        model.get_input_embeddings().weight[token_id] = torch.tensor(init_vec, dtype=torch.bfloat16)

print(f"✅ Embeddings initialised (geometric sinusoidal)")

# Verify roundtrip
test_tok = "[ROUTE_TIME_ADD]"
tid = tokenizer.convert_tokens_to_ids(test_tok)
print(f"Token '{test_tok}' → id={tid} → '{tokenizer.convert_ids_to_tokens(tid)}'")

## 6. Load & Tokenize Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': FORMATTED_TRAIN,
    'eval': FORMATTED_EVAL,
})

print(f"Train: {len(dataset['train'])} samples")
print(f"Eval:  {len(dataset['eval'])} samples")
print(f"\nSample:")
print(dataset['train'][0]['text'][:400])

In [ ]:
MAX_SEQ_LEN = 384  # Llama headers are longer than SmolLM2's ChatML

def tokenize_fn(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

print(f"Tokenized train: {len(tokenized['train'])}")
print(f"Tokenized eval:  {len(tokenized['eval'])}")

sample_nonpad = (tokenized['train'][0]['input_ids'] != tokenizer.pad_token_id).sum().item()
print(f"Sample sequence length (non-pad): {sample_nonpad}")

truncated = sum(1 for i in range(min(1000, len(tokenized['train'])))
                if (tokenized['train'][i]['input_ids'] != tokenizer.pad_token_id).sum().item() >= MAX_SEQ_LEN)
print(f"Truncated samples (first 1000): {truncated} — should be 0 or near-0")

## 7. Training Configuration

**Key decisions:**
- **5K steps** (not 10K) — 1B model learns faster, reduces catastrophic forgetting risk
- **bf16** — Llama was pre-trained in bf16, L4 has native bf16 support
- **gradient_checkpointing** — trades compute for VRAM (1B is tight otherwise)
- **batch 2 × 8 grad_accum = 16 effective** — larger model needs smaller per-device batch
- **adamw_8bit** — saves ~3GB VRAM on optimiser states
- **LR 5e-5** — lower than SmolLM2's 1e-4 (larger models need gentler updates)
- **Checkpoint every 1K steps** — for interval validation with the harness

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=5000,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    optim="adamw_8bit",
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=6,
    logging_steps=25,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="tensorboard",
    dataloader_num_workers=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
total_samples_seen = training_args.max_steps * effective_batch
epochs_approx = total_samples_seen / len(tokenized['train'])

print(f"Effective batch size: {effective_batch}")
print(f"Total samples seen: {total_samples_seen:,}")
print(f"Approximate epochs: {epochs_approx:.2f}")
print(f"Checkpoints: every {training_args.save_steps} steps")
print(f"LR: {training_args.learning_rate}")
print(f"Optimiser: {training_args.optim}")

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=data_collator,
    processing_class=tokenizer,
)

mem_gb = torch.cuda.memory_allocated() / 1e9
max_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ Trainer ready")
print(f"VRAM: {mem_gb:.1f} GB used / {max_gb:.1f} GB total ({mem_gb/max_gb*100:.0f}%)")

## 8. Train!

Expected runtime on L4: **~2-3 hours** for 5K steps.

Watch for:
- Train loss should drop below 0.5 by step 1000
- Eval loss should track train loss (no divergence = no overfitting)
- If eval loss starts rising while train loss drops, stop early

In [ ]:
train_result = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"  Samples/sec: {train_result.metrics['train_samples_per_second']:.1f}")

## 9. Evaluate

In [ ]:
eval_results = trainer.evaluate()
print(f"\nEval loss: {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## 10. Quick Inference Test

**Llama 3.2 EOS:** `<|eot_id|>` (not `<|im_end|>` like SmolLM2)

In [ ]:
import sys
sys.path.insert(0, 'tagzeit-gemma-2b')
from tools.validate import SYSTEM_PROMPT

test_prompts = [
    "What time is it 1 minute after 23:59?",
    "What time was it 30 minutes before 00:15?",
    "How much time is there between 09:00 and 17:30?",
    "The meeting starts at 14:30 and lasts 45 minutes. When does it end?",
    "I arrived at work at 09:15. The commute was 25 minutes. When did I leave home?",
    "What is 42 + 18?",
    "When was The Secret Garden published?",
    "What timezone is Tokyo in?",
]

model.eval()
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=1.0,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    response = response.split('<|eot_id|>')[0].strip()

    print(f"Q: {prompt}")
    print(f"A: {response}")
    print()

## 11. Save Final Model

In [ ]:
trainer.save_model(f"{OUTPUT_DIR}/best")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
print(f"✅ Best model saved to {OUTPUT_DIR}/best")

import glob
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"\nCheckpoints saved:")
for cp in checkpoints:
    print(f"  {cp}")

## 12. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs

---

## Resume from Checkpoint

If Colab disconnects, re-run cells 1–7 to rebuild everything, then:

In [ ]:
# Re-run cells 1-7 first, then:
# train_result = trainer.train(resume_from_checkpoint=True)
#
# Or from a specific checkpoint:
# train_result = trainer.train(resume_from_checkpoint=f"{OUTPUT_DIR}/checkpoint-2000")